In [ ]:
# Cell 0 — Install Unsloth (cu unsloth_zoo)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA capability: {torch.cuda.get_device_capability()}")

# Install Unsloth + dependențe (inclusiv unsloth_zoo!)
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers
!pip install -q -U datasets

print("\n=== Pachete instalate ===")
!pip list 2>/dev/null | grep -i -E "^(unsloth|unsloth_zoo|unsloth-zoo|xformers|trl|peft|bitsandbytes|accelerate)"

print("\n!!! Restart & clear cell outputs ACUM, apoi rulează Cell 1 !!!")

In [1]:
import os, shutil

# Curățare AGRESIVĂ — toate cache-urile posibile
for path in [
    "/kaggle/working/unsloth_compiled_cache",
    "/tmp/torchinductor_root",
    "/root/.cache/torch/inductor",
    os.path.expanduser("~/.cache/torch/inductor"),
    "/kaggle/working/.cache",
]:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f"Șters: {path}")
    else:
        print(f"Nu există: {path}")

# Dezactivează cache la PyTorch
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/no_cache_" + str(os.getpid())
os.environ["UNSLOTH_DISABLE_COMPILE"] = "1"   # CRITIC: dezactivează compile-ul Unsloth complet

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc
gc.collect()
torch.cuda.empty_cache()

from unsloth import FastLanguageModel

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Nu există: /kaggle/working/unsloth_compiled_cache
Nu există: /tmp/torchinductor_root
Nu există: /root/.cache/torch/inductor
Nu există: /root/.cache/torch/inductor
Nu există: /kaggle/working/.cache
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
VRAM allocated: 0.01 GB


In [2]:
# Cell 2 — locate the dataset
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/ematoda/fanfictionft/system_default.txt
/kaggle/input/datasets/ematoda/fanfictionft/train_merged.jsonl
/kaggle/input/datasets/ematoda/fanfictionft/eval_merged.jsonl


In [3]:
# Cell 3 — config + load data

import json
from pathlib import Path
from datasets import Dataset

# ── Paths
DATA_DIR   = Path("/kaggle/input/datasets/ematoda/fanfictionft")
OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE         = DATA_DIR / "train_merged.jsonl"
EVAL_FILE          = DATA_DIR / "eval_merged.jsonl"
SYSTEM_PROMPT_PATH = DATA_DIR / "system_default.txt"

assert TRAIN_FILE.exists()
assert SYSTEM_PROMPT_PATH.exists()

SYSTEM_PROMPT = SYSTEM_PROMPT_PATH.read_text(encoding="utf-8").strip()
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_raw = load_jsonl(TRAIN_FILE)
eval_raw  = load_jsonl(EVAL_FILE)
print(f"Train: {len(train_raw)} | Eval: {len(eval_raw)}")


System prompt: 2189 chars
Train: 6523 | Eval: 342


In [4]:
max_seq_length = 1024
dtype          = None
load_in_4bit   = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "NousResearch/Hermes-3-Llama-3.2-3B",
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)

print(f"\nModel loaded.")
print(f"VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

Unsloth: Will load NousResearch/Hermes-3-Llama-3.2-3B as a legacy tokenizer.



Model loaded.
VRAM after load: 2.29 GB


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,                                       # ← 8 → 16
    lora_alpha     = 32,                                       # ← 16 → 32
    lora_dropout   = 0,
    bias           = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",  # ← toate 7 module
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
    use_rslora                 = False,
    loftq_config               = None,
)

print(f"VRAM after LoRA: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


VRAM after LoRA: 2.39 GB


In [6]:
# Cell 6 — format data în template Hermes 3 (ChatML)

# Hermes 3 folosește ChatML
def format_chatml(row):
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{row['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['output']}<|im_end|>"
    )
    return {"text": text}

train_dataset = Dataset.from_list([format_chatml(r) for r in train_raw])
eval_dataset  = Dataset.from_list([format_chatml(r) for r in eval_raw])

print(f"Sample (first 400 chars):\n{train_dataset[0]['text'][:400]}")


Sample (first 400 chars):
<|im_start|>system
You are an expert fiction writer specializing in short stories. Your sole task is to write original, compelling short stories in response to user prompts.

OUTPUT FORMAT:
- Begin with a title on the first line, formatted as: Title: [Your Title]
- Follow immediately with the story, no preamble, no commentary, no explanation
- Do not include any text after the story ends

TONE AND


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,    # ← 1 → 4
        gradient_accumulation_steps = 4,    # ← 16 → 4 (effective batch tot 16)
        warmup_steps                = 5,
        num_train_epochs            = 1,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = str(OUTPUT_DIR),
        save_strategy               = "steps",
        save_steps                  = 200,
        save_total_limit            = 2,
        report_to                   = "none",
    ),
)
print("Trainer ready.")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/6523 [00:00<?, ? examples/s]

Trainer ready.
VRAM: 2.39 GB


In [9]:
# Cell 8 — TRAIN cu cleanup VRAM înainte
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"VRAM înainte de train: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Starting training...", flush=True)

trainer_stats = trainer.train()
print("\nTraining complete.")
print(f"Time: {trainer_stats.metrics['train_runtime']:.1f}s")

VRAM înainte de train: 2.39 GB
Starting training...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128039}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,523 | Num Epochs = 1 | Total steps = 408
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.724558
10,2.091219
15,1.539932
20,1.487348
25,1.480749
30,1.476260
35,1.461590
40,1.416055
45,1.447894
50,1.431655


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/output/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/output/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/output/checkpoint-408/tokenizer_config.json.



Training complete.
Time: 6293.7s


In [10]:
# Cell 9 — save adapter + quick test
import shutil

ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

shutil.make_archive("/kaggle/working/final_adapter", "zip", str(ADAPTER_DIR))
print(f"Saved & zipped: /kaggle/working/final_adapter.zip")

# Quick inference test
FastLanguageModel.for_inference(model)   # 2x speed pentru inference

def generate(prompt, max_new=400):
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=max_new,
        temperature=0.8, top_p=0.9, repetition_penalty=1.1,
        do_sample=True, use_cache=True,
    )
    return tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True)[0]

print(generate("Write a short story about an astronaut on Mars."))


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/output/final_adapter/tokenizer_config.json.


Saved & zipped: /kaggle/working/final_adapter.zip


Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

The airlock hummed softly as I entered. The suit, all black and rigid, seemed uncomfortable. Not that there was anything else to wear, but still, it felt like I had been stuffed into it. I checked my gear, making sure nothing was loose or out of place, then sat down in front of the small console where my flight computer was waiting.

It was just me, now. At least at this stage, anyway. In a few minutes, the other three would join me on the surface. One of them was a woman named Sarah, who was going to be the commander. She was tall and athletic and had already proven herself quite capable on previous missions. Another was a man named Dan, a biologist. He was shorter than me by several inches and had glasses, which I found mildly amusing. The third was Jai, the pilot. They were all young - we'd made sure they knew how to handle themselves before they left Earth. But they weren't ready yet, so I'd have to hold their hands until they got used to everything.

As the suit beeped and whirred

In [ ]:
import os, gc, torch

# Curăță VRAM înainte
gc.collect()
torch.cuda.empty_cache()

# 1. Încarcă baza + adapterul TĂU (calea corectă din notebook-ul tău)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="NousResearch/Hermes-3-Llama-3.2-3B",  # baza
    max_seq_length=1024,   # același ca la training
    dtype=None,
    load_in_4bit=False,    # False pentru merge corect
)

from peft import PeftModel
model = PeftModel.from_pretrained(
    model, 
    "/kaggle/working/output/final_adapter"  # adapterul tău
)

# 2. Merge
model = model.merge_and_unload()
print("Merge done!")

# 3. Salvează modelul complet
model.save_pretrained("/kaggle/working/final_merged_model")
tokenizer.save_pretrained("/kaggle/working/final_merged_model")
print("Salvat la /kaggle/working/final_merged_model")

# 4. Arhivează pentru download
import shutil
shutil.make_archive("/kaggle/working/final_merged_model", "zip", "/kaggle/working/final_merged_model")
print("ZIP creat: /kaggle/working/final_merged_model.zip")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load NousResearch/Hermes-3-Llama-3.2-3B as a legacy tokenizer.


Merge done!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/final_merged_model/tokenizer_config.json.


Salvat la /kaggle/working/final_merged_model
